In [1]:
import pathlib

import yaml
import pandas as pd

import jax
import jax.numpy as jnp
import jax.random as jr

import torch

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core

import ipywidgets as widgets
from sklearn.manifold import TSNE

# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

So that `impor`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The directory containing the data

In [3]:
data_directory = pathlib.Path.cwd().parent / 'data'
assert data_directory.exists()

The particular dataset (specified by a *base* file name)

In [4]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = data_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = data_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

Dataset is loaded

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
len(ds)

12983

The directory containing the actual results

In [6]:
results_directory = pathlib.Path.cwd()
assert results_directory.exists()

*Base* file name for the results (trained model)

In [7]:
# input_files_stem = pathlib.Path('/tmp') / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_500e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z20_300e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_10_300_epochs' 
input_files_stem = results_directory / 'new' / 'z_50_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_2_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_3_300_epochs' 


In [8]:
# w_file_upload = widgets.FileUpload(
#     accept='.eqx',
#     multiple=False
# )
# display(w_file_upload)

In [9]:
# w_file_upload.value

From the above *stem*, several file names are derived

In [10]:
# parameters_file = pathlib.Path('parameters.yaml')
parameters_file = input_files_stem.with_suffix('.yaml')
assert parameters_file.exists()

# weights_file = pathlib.Path('foo.eqx')
weights_file = input_files_stem.with_suffix('.eqx')
assert weights_file.exists()

results_file = input_files_stem.with_suffix('.parquet')
assert results_file.exists()

In [11]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
parameters

{'PRNG seeds': {'model initialization': {'decoder': 43, 'encoder': 42},
  'sampling': {'training set': 14, 'validation set': 15},
  'training dataloader': 43,
  'training-validation splitting': 42},
 'encoder': {'input_size': [1, 10000]},
 'loss': {'x_sd': 0.25},
 'model': {'channels': [12, 12, 12],
  'dim_z': 50,
  'kernel_sizes': [128, 64, 32],
  'stride': 2},
 'training': {'batch_size': 32,
  'dl_rng_key': 43,
  'learning_rate': 3e-05,
  'n_epochs': 300,
  'train_test_split': {'test_size': 0.2}}}

In [12]:
results_df = pd.read_parquet(results_file)
results_df

training                              validation                \
             loss loglikelihood          kl          loss loglikelihood   
0    53285.791412 -53123.421280  162.370163  30294.680807 -30027.265006   
1    27905.011719 -27593.666367  311.345325  26352.327696 -26004.388481   
2    25648.908918 -25288.733212  360.175734  25146.346608 -24773.675877   
3    24702.890757 -24329.036472  373.854259  24469.998499 -24091.930688   
4    24190.128867 -23815.185685  374.943163  24065.801996 -23692.647771   
..            ...           ...         ...           ...           ...   
295  18379.622800 -18133.303642  246.319160  19344.585604 -19101.085640   
296  18392.093422 -18146.806379  245.287084  19240.253216 -18997.212140   
297  18384.647581 -18139.175162  245.472415  19334.408787 -19090.609685   
298  18355.284202 -18110.638202  244.646001  19285.562286 -19041.268376   
299  18356.021770 -18111.253756  244.768032  19268.867211 -19027.882277   

                 
             kl  
0    267.415786  
1    347.939173  
2    372.670750  
3    378.067958  
4    373.154231  
..          ...  
295  243.499998  
296  243.041009  
297  243.799060  
298  244.293935  
299  240.984892  

[300 rows x 6 columns]

In [13]:
def metric_chart(df: pd.DataFrame, metric: str):

    sub_df = df.loc[:,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

alt.HConcatChart(...)

In [14]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [15]:
sum(trapallada.util.get_layers_n_parameters(model).values())

2593541

Data loaders for both the training and validation sets

In [16]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])

A batch from the **validation** set

In [17]:
# batch = next(iter(train_dl))
batch = next(iter(validation_dl))

In [18]:
# ds.metadata_df

An element from the above batch is encoded, decoded, and a prediction plotted

In [19]:
# i = 0
# i = 1
i = 25

i_s, x_s, y_s = batch
x_s = x_s.numpy()
x_pred, z_mean, z_sd = model(x_s[i], jr.PRNGKey(42))

display(ds.metadata_df.loc[i_s[i].item()])

# first dimension of `x_pred` is 1, and hence it's squeezed out
df = pd.DataFrame({'prediction': x_pred[0], 'actual': x_s[i].ravel()})
n_rows = 2000
alt.Chart(
    df.iloc[:n_rows].reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

label                                 FM
SNR                                3.162
color                               None
file name    FM/FM_SNR3.162_Simul664.mat
Name: 2552, dtype: object

alt.Chart(...)

In [20]:
z_mean, z_sd

(Array([ 9.0892416e-01, -3.0295203e+00,  3.5027260e-01, -1.6428545e+00,
        -1.2641844e+00, -3.1973532e-01,  1.2142084e-01,  4.0501618e-01,
         2.3701215e+00,  2.8033969e+00,  3.2781267e+00,  3.5931563e-01,
        -1.1001350e+00, -1.0564542e+00, -1.2125755e-01,  9.7100502e-01,
         1.4496276e-01,  3.6949043e+00, -1.3217435e+00,  2.8191623e-01,
         1.6639142e+00, -1.7853751e+00,  1.5532608e+00, -2.0609162e+00,
         2.0465236e+00, -3.6297252e+00, -1.7647793e+00, -3.0640063e+00,
        -2.6856375e+00,  1.4834256e-01,  1.3747585e+00, -1.8901572e+00,
         1.6490878e+00,  2.7129604e-03, -1.9139279e+00,  3.2008395e+00,
         5.0912285e-01,  5.7847059e-01, -2.7121079e+00,  1.4503566e+00,
         3.9448309e+00, -3.0454108e-04, -2.4948647e+00, -3.2152057e+00,
        -5.4616588e-01, -5.8858854e-01,  4.9011266e-01,  5.6632823e-01,
         2.3385857e-01, -1.0792105e-02], dtype=float32),
 Array([0.02639751, 0.03378533, 0.03002854, 0.03463432, 0.02817997,
        0.0

In [21]:
df = trapallada.util.make_projections_df(ds, validation_dl, model.encoder)
# df = trapallada.util.make_projections_df(ds, train_dl, model.encoder)
df.head()

,label,SNR,color,file name,z_1,z_2,z_3,z_4,z_5,z_6,...,z_43,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2
8940,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul931.mat,0.955214,-0.744425,0.603479,-0.056048,-1.294133,0.475934,...,-0.871829,-1.917214,-0.987643,-1.813413,0.163278,0.348527,0.572991,1.267724,8.662113,18.811094
7298,AM,31.620,None,AM/AM_SNR31.62_Simul641.mat,-1.608342,-0.478551,-0.278583,0.807158,-0.615133,-0.450265,...,0.151691,-3.208567,-2.095595,0.373494,1.330342,0.287232,2.202979,1.088996,-28.145939,3.907068
10764,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul853.mat,0.134577,-0.942041,2.522903,2.209447,-0.619013,-1.334488,...,0.288943,-2.610679,-0.728518,-1.346779,-0.906536,1.157483,1.510882,0.788501,5.789162,-37.197994
1331,FM,31.620,None,FM/FM_SNR31.62_Simul524.mat,-0.118946,-1.463893,-0.497641,1.671303,-1.558171,-0.862737,...,-2.089194,-1.416743,-0.072907,-1.834596,0.990254,1.905218,0.553023,-0.251939,2.067066,-2.760794
10608,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul60.mat,0.151970,-0.215203,1.362321,0.446605,-2.421160,2.235139,...,-2.658431,-5.135908,-1.863745,-1.554168,0.882697,1.772723,1.167415,0.876318,-42.801983,-45.259216


Signals that are just noise have no SNR, but that's fine for...

In [22]:
df['SNR'].isna().sum()

197

...plotting

In [23]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='label:N',
    fill='label:N',
    size='SNR:Q',
    tooltip=['file name:N']
).properties(
    width=800, height=600
).interactive()

/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

The latent space is projected into a 3D space

In [24]:
df_3 = trapallada.util.make_projections_df(ds, validation_dl, model.encoder, n_projected_components=3)
df_3.head()

,label,SNR,color,file name,z_1,z_2,z_3,z_4,z_5,z_6,...,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2,tsne_3
8940,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul931.mat,0.955214,-0.744425,0.603479,-0.056048,-1.294133,0.475934,...,-1.917214,-0.987643,-1.813413,0.163278,0.348527,0.572991,1.267724,8.686266,5.820134,-0.389920
7298,AM,31.620,None,AM/AM_SNR31.62_Simul641.mat,-1.608342,-0.478551,-0.278583,0.807158,-0.615133,-0.450265,...,-3.208567,-2.095595,0.373494,1.330342,0.287232,2.202979,1.088996,-11.132998,6.707359,3.389960
10764,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul853.mat,0.134577,-0.942041,2.522903,2.209447,-0.619013,-1.334488,...,-2.610679,-0.728518,-1.346779,-0.906536,1.157483,1.510882,0.788501,4.167854,-13.031677,1.361701
1331,FM,31.620,None,FM/FM_SNR31.62_Simul524.mat,-0.118946,-1.463893,-0.497641,1.671303,-1.558171,-0.862737,...,-1.416743,-0.072907,-1.834596,0.990254,1.905218,0.553023,-0.251939,-0.243868,-6.834010,-3.830881
10608,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul60.mat,0.151970,-0.215203,1.362321,0.446605,-2.421160,2.235139,...,-5.135908,-1.863745,-1.554168,0.882697,1.772723,1.167415,0.876318,4.230256,-17.624647,9.266415


In [25]:
import plotly.express as px
fig = px.scatter_3d(df_3, x='tsne_1', y='tsne_2', z='tsne_3', color='label', opacity=0.5, width=800, height=800)
fig.show()